# Horse-racing betting model

This notebook takes the cleaned dataset from the EDA and builds a set of probability models for predicting whether a horse wins.

The workflow is:

1. Prepare the modelling dataset
2. Split chronologically to avoid future information leaking into training
3. Build preprocessing pipelines
4. Fit a logistic-regression baseline
5. Fit XGBoost
6. Compare model probabilities with the betting market
7. Evaluate calibration
8. Tune XGBoost using time-series cross-validation
9. Evaluate the resulting betting strategy
10. Inspect feature importance

The main prediction target is `won`, and the main modelling objective is **probability quality**, so log loss and Brier score are prioritised over accuracy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

## 1. Load the modelling data

This notebook assumes the merged and feature-engineered dataset has already been prepared in the EDA notebook.

If you run this notebook independently, replace the path below with the location of your cleaned modelling dataset.

In [ ]:
# Use the same merged dataset produced in the EDA
# Adjust this path if your cleaned dataset is saved elsewhere.

runs = pd.read_csv("../data/runs.csv")
races = pd.read_csv("../data/races.csv")

df = runs.merge(
    races,
    on="race_id",
    how="left",
    validate="many_to_one"
)

df["date"] = pd.to_datetime(df["date"])

# Historical features must be calculated using only races that occurred previously.
df = df.sort_values(["date", "race_id", "horse_id"]).reset_index(drop=True)

# Historical horse/jockey/trainer features
df["horse_prior_runs"] = df.groupby("horse_id").cumcount()
df["horse_prior_wins"] = df.groupby("horse_id")["won"].cumsum() - df["won"]
df["horse_prior_win_rate"] = (
    df["horse_prior_wins"] / df["horse_prior_runs"].replace(0, np.nan)
)

df["jockey_prior_runs"] = df.groupby("jockey_id").cumcount()
df["jockey_prior_wins"] = df.groupby("jockey_id")["won"].cumsum() - df["won"]
df["jockey_prior_win_rate"] = (
    df["jockey_prior_wins"] / df["jockey_prior_runs"].replace(0, np.nan)
)

df["trainer_prior_runs"] = df.groupby("trainer_id").cumcount()
df["trainer_prior_wins"] = df.groupby("trainer_id")["won"].cumsum() - df["won"]
df["trainer_prior_win_rate"] = (
    df["trainer_prior_wins"] / df["trainer_prior_runs"].replace(0, np.nan)
)

df.head()

## 2. Define the features

Keep the feature lists explicit. This makes it much easier to understand exactly what information the model is allowed to use and to compare different model specifications later.

In [ ]:
numeric_features = [
    "horse_age",
    "declared_weight",
    "actual_weight",
    "draw",
    "win_odds",
    "place_odds",
    "race_no",
    "distance",
    "prize",
    "horse_prior_runs",
    "horse_prior_win_rate",
    "jockey_prior_runs",
    "jockey_prior_win_rate",
    "trainer_prior_runs",
    "trainer_prior_win_rate",
]

categorical_features = [
    "venue",
    "surface",
    "going",
    "horse_sex",
    "horse_country",
    "horse_gear",
    "config",
    "race_class",
]

target = "won"

feature_columns = categorical_features + numeric_features

### Encode multi-valued horse gear

`horse_gear` can contain several pieces of equipment separated by `/`. We therefore represent each individual item as a binary feature rather than treating the complete string as one category.

In [ ]:
gear_dummies = (
    df["horse_gear"]
    .fillna("")
    .str.get_dummies(sep="/")
    .add_prefix("gear_")
)

df = pd.concat([df, gear_dummies], axis=1)

categorical_features = categorical_features + gear_dummies.columns.tolist()
feature_columns = categorical_features + numeric_features

print(f"Number of numerical features: {len(numeric_features)}")
print(f"Number of categorical features: {len(categorical_features)}")

## 3. Chronological train/test split

For a betting problem, a random train/test split would allow the model to learn from races that occurred after the races it is being evaluated on.

Instead, the test set contains the most recent 20% of observations.

In [ ]:
split_date = df["date"].quantile(0.80)

train = df[df["date"] < split_date].copy()
test = df[df["date"] >= split_date].copy()

print(f"Split date: {split_date.date()}")
print(f"Training observations: {len(train):,}")
print(f"Test observations: {len(test):,}")

print("\nTrain period:")
print(train["date"].min(), "to", train["date"].max())

print("\nTest period:")
print(test["date"].min(), "to", test["date"].max())

## 4. Build the preprocessing pipeline

The preprocessing is kept inside the sklearn pipeline so that transformations are fitted **only on the training data**.

- Numerical variables: median imputation → standardisation
- Categorical variables: most-frequent imputation → one-hot encoding

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

X_train = train[feature_columns]
X_test = test[feature_columns]

y_train = train[target]
y_test = test[target]

## 5. Logistic regression baseline

Logistic regression gives us a simple baseline and is useful for checking whether the more complex models are actually adding predictive value.

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000)),
])

logistic_model.fit(X_train, y_train)

lr_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Log loss:", log_loss(y_test, lr_prob))
print("Brier score:", brier_score_loss(y_test, lr_prob))
print("Accuracy:", logistic_model.score(X_test, y_test))

### Logistic-regression coefficients

The coefficients are useful for interpretation, although they should not be treated as causal effects.

In [ ]:
feature_names = logistic_model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = logistic_model.named_steps["classifier"].coef_[0]

coef_df = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
    })
    .sort_values("coefficient", ascending=False)
)

display(coef_df.head(20))
display(coef_df.tail(20))

## 6. XGBoost baseline

XGBoost can capture nonlinear relationships and interactions that logistic regression cannot.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model),
])

xgb_pipeline.fit(X_train, y_train)

xgb_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

print("Log loss:", log_loss(y_test, xgb_prob))
print("Brier score:", brier_score_loss(y_test, xgb_prob))

## 7. Compare model probabilities with the market

For decimal odds, the raw implied probability is `1 / odds`.

Because the raw probabilities contain the bookmaker's margin, we normalise them within each race so that the probabilities sum to 1.

In [ ]:
df["market_raw_prob"] = 1 / df["win_odds"]

df["market_prob"] = (
    df["market_raw_prob"]
    / df.groupby("race_id")["market_raw_prob"].transform("sum")
)

market_prob = test["market_prob"]

print("Market probability check:")
print(
    df.groupby("race_id")["market_prob"]
      .sum()
      .head()
)

print("\nMarket log loss:", log_loss(y_test, market_prob))
print("Market Brier score:", brier_score_loss(y_test, market_prob))

## 8. Calibration

For betting, a probability of 0.20 should correspond approximately to a 20% empirical win rate over many comparable predictions.

We therefore compare the calibration of:
- Logistic regression
- XGBoost
- Market probabilities

In [ ]:
lr_true, lr_pred = calibration_curve(
    y_test, lr_prob, n_bins=10
)

xgb_true, xgb_pred = calibration_curve(
    y_test, xgb_prob, n_bins=10
)

market_true, market_pred = calibration_curve(
    y_test, market_prob, n_bins=10
)

plt.figure(figsize=(8, 6))

plt.plot(lr_pred, lr_true, marker="o", label="Logistic Regression")
plt.plot(xgb_pred, xgb_true, marker="o", label="XGBoost")
plt.plot(market_pred, market_true, marker="o", label="Market")
plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed win rate")
plt.title("Calibration Curve")
plt.legend()
plt.show()

## 9. Evaluate model edge and expected value

The model is potentially useful for betting only if it identifies horses whose estimated probability is sufficiently higher than the market's implied probability.

Two useful quantities are:

- **Probability edge:** model probability − market probability
- **Expected value:** `model probability × decimal odds − 1`

An EV above zero means the model estimates a positive expected return before considering additional costs or practical betting constraints.

In [ ]:
test = test.copy()

test["model_prob_xgb"] = xgb_prob
test["prob_edge"] = test["model_prob_xgb"] - test["market_prob"]

test["ev"] = (
    test["model_prob_xgb"] * test["win_odds"] - 1
)

display(
    test.sort_values("ev", ascending=False)[
        [
            "race_id",
            "horse_id",
            "win_odds",
            "model_prob_xgb",
            "market_prob",
            "prob_edge",
            "ev",
            "won",
        ]
    ].head(20)
)

## 10. Simple betting-strategy backtest

We test a simple strategy: bet £1 whenever the model's estimated EV exceeds a chosen threshold.

This is deliberately a basic backtest. It does not account for factors such as bet limits, changing odds, transaction costs, or whether the quoted odds were actually available at the time of betting.

In [ ]:
test["profit"] = np.where(
    test["won"] == 1,
    test["win_odds"] - 1,
    -1,
)

thresholds = [0, 0.05, 0.10, 0.20, 0.50]

results = []

for threshold in thresholds:
    bets = test[test["ev"] > threshold]

    profit = bets["profit"].sum()
    staked = len(bets)

    results.append({
        "ev_threshold": threshold,
        "bets": staked,
        "profit": profit,
        "roi": profit / staked if staked else np.nan,
    })

betting_results = pd.DataFrame(results)
display(betting_results)

## 11. Calibrated XGBoost

XGBoost probabilities are not necessarily well calibrated. We can therefore compare the original model with a calibrated version.

Note that this calibration step should ideally be fitted using data separate from the final test set; the test set remains untouched for final evaluation.

In [ ]:
xgb_calibrated = CalibratedClassifierCV(
    xgb_pipeline,
    method="sigmoid",
    cv=5,
)

xgb_calibrated.fit(X_train, y_train)

xgb_cal_prob = xgb_calibrated.predict_proba(X_test)[:, 1]

print("Uncalibrated XGBoost")
print("Log loss:", log_loss(y_test, xgb_prob))
print("Brier score:", brier_score_loss(y_test, xgb_prob))

print("\nCalibrated XGBoost")
print("Log loss:", log_loss(y_test, xgb_cal_prob))
print("Brier score:", brier_score_loss(y_test, xgb_cal_prob))

In [ ]:
calibrated_true, calibrated_pred = calibration_curve(
    y_test,
    xgb_cal_prob,
    n_bins=10,
)

plt.figure(figsize=(8, 6))

plt.plot(lr_pred, lr_true, marker="o", label="Logistic Regression")
plt.plot(xgb_pred, xgb_true, marker="o", label="XGBoost")
plt.plot(market_pred, market_true, marker="o", label="Market")
plt.plot(
    calibrated_pred,
    calibrated_true,
    marker="o",
    label="Calibrated XGBoost",
)

plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed win rate")
plt.title("Calibration Comparison")
plt.legend()
plt.show()

## 12. Tune XGBoost with time-series cross-validation

The hyperparameters are selected using `TimeSeriesSplit`, rather than random cross-validation.

The scoring metric is **negative log loss**, because the goal is to produce useful probabilities rather than simply classify winners correctly.

In [ ]:
xgb_search_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        random_state=42,
    )),
])

param_grid = {
    "model__n_estimators": [200, 500],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [3, 6],
    "model__min_child_weight": [1, 5],
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=xgb_search_pipeline,
    param_grid=param_grid,
    scoring="neg_log_loss",
    cv=tscv,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest CV log loss:")
print(-grid_search.best_score_)

## 13. Evaluate the tuned XGBoost on the held-out test set

In [ ]:
best_xgb = grid_search.best_estimator_

tuned_xgb_prob = best_xgb.predict_proba(X_test)[:, 1]

print("Tuned XGBoost")
print("Log loss:", log_loss(y_test, tuned_xgb_prob))
print("Brier score:", brier_score_loss(y_test, tuned_xgb_prob))

print("\nMarket")
print("Log loss:", log_loss(y_test, market_prob))
print("Brier score:", brier_score_loss(y_test, market_prob))

In [ ]:
tuned_true, tuned_pred = calibration_curve(
    y_test,
    tuned_xgb_prob,
    n_bins=10,
)

plt.figure(figsize=(8, 6))

plt.plot(lr_pred, lr_true, marker="o", label="Logistic Regression")
plt.plot(xgb_pred, xgb_true, marker="o", label="XGBoost")
plt.plot(tuned_pred, tuned_true, marker="o", label="Tuned XGBoost")
plt.plot(market_pred, market_true, marker="o", label="Market")
plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed win rate")
plt.title("Final Calibration Comparison")
plt.legend()
plt.show()

## 14. Betting performance of the tuned model

In [ ]:
test["model_prob"] = tuned_xgb_prob

test["prob_edge"] = (
    test["model_prob"] - test["market_prob"]
)

test["ev"] = (
    test["model_prob"] * test["win_odds"] - 1
)

test["profit"] = np.where(
    test["won"] == 1,
    test["win_odds"] - 1,
    -1,
)

thresholds = [0, 0.05, 0.10, 0.20, 0.50]

results = []

for threshold in thresholds:
    bets = test[test["ev"] > threshold]

    profit = bets["profit"].sum()
    staked = len(bets)

    results.append({
        "ev_threshold": threshold,
        "bets": staked,
        "profit": profit,
        "roi": profit / staked if staked else np.nan,
    })

display(pd.DataFrame(results))

### Where does the model find an edge?

Looking at the distribution of model-vs-market differences helps assess whether the model is systematically finding opportunities or simply producing noisy deviations from the market.

In [ ]:
test["edge_bin"] = pd.qcut(
    test["prob_edge"],
    q=10,
    duplicates="drop",
)

edge_summary = (
    test.groupby("edge_bin", observed=True)
    .agg(
        n=("won", "size"),
        model_prob=("model_prob", "mean"),
        market_prob=("market_prob", "mean"),
        actual_win_rate=("won", "mean"),
        mean_ev=("ev", "mean"),
        roi=("profit", "mean"),
    )
)

display(edge_summary)

## 15. Performance by odds range

In [ ]:
test["odds_bin"] = pd.cut(
    test["win_odds"],
    bins=[0, 2, 5, 10, 20, 50, 100],
    include_lowest=True,
)

odds_summary = (
    test.groupby("odds_bin", observed=True)
    .agg(
        n=("won", "size"),
        model_prob=("model_prob", "mean"),
        actual_win_rate=("won", "mean"),
        mean_odds=("win_odds", "mean"),
        mean_ev=("ev", "mean"),
        roi=("profit", "mean"),
    )
)

display(odds_summary)

## 16. Feature importance

Permutation importance measures how much predictive performance deteriorates when a feature is shuffled.

We use log loss here because probability quality is the main objective.

In [ ]:
importance_result = permutation_importance(
    best_xgb,
    X_test,
    y_test,
    scoring="neg_log_loss",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance": importance_result.importances_mean,
    })
    .sort_values("importance", ascending=False)
)

display(importance_df.head(20))

In [ ]:
(
    importance_df.head(20)
    .sort_values("importance")
    .plot(
        x="feature",
        y="importance",
        kind="barh",
        figsize=(8, 6),
        legend=False,
    )
)

plt.xlabel("Mean decrease in performance")
plt.ylabel("")
plt.title("XGBoost Permutation Importance")
plt.show()

## 17. Next modelling experiments

The current notebook establishes a clean baseline. Useful next experiments are:

- **Market-independent model:** remove `win_odds` and `place_odds` to test whether the model contains information beyond the market.
- **Alternative models:** LightGBM, CatBoost, Random Forest, Extra Trees and HistGradientBoosting.
- **Race-level evaluation:** evaluate whether the model identifies the winner within each race rather than treating every horse independently.
- **Probability normalisation:** investigate whether model probabilities should be normalised within each race.
- **Better validation:** use rolling/expanding time-based validation for model selection.
- **More realistic betting backtest:** account for odds availability, staking rules and possible limits.
- **Calibration:** compare Platt/sigmoid calibration with isotonic calibration using a proper validation period.

The most important next experiment is probably the **market-independent model**, because it tells us whether the predictive signal is genuinely adding information beyond the betting market.